In [1]:
!pip install transformers==4.36.0 -q
!pip install multilingual-clip -q
!pip install easyocr -q
!pip install open_clip_torch -q
!pip install POT

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 38.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.7.0 requires tokenizers>=0.19, but you have tokenizers 0.15.2 which is incompatible.
sentence-transformers 5.7.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.0 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 

In [2]:
import os
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ot

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import transformers
import open_clip

from scipy.ndimage import gaussian_filter
import cv2
import matplotlib.cm as cm

import easyocr
import matplotlib.pyplot

/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [3]:
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}")

transformers: 4.36.0
torch: 2.11.0+cpu


In [4]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [5]:
#output_folder = "SALIENCY_MAPS_FULL_FINETUNED"
#output_folder = "SALIENCY_MAPS_LAST_LAYER_FINETUNED"
output_folder='/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-11] full_fine'  #[N] in base al layer cui si vuole lavorare
#image_folder = "/scratch_share/mind/claudia/MEME_UNIVOCI"
image_folder='/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/Dati/MCLIP/MEME_UNIVOCI'
#text_df=pd.read_csv('trascrizioni_ocr.csv')
text_df = pd.read_csv('/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/Dati/MCLIP/OCR/TRASCRIZIONI.csv')
os.makedirs(output_folder, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [6]:
class MultimodalClassifier(nn.Module):
    def __init__(self, image_model, text_model, tokenizer):
        super().__init__()
        self.image_model = image_model
        self.text_model = text_model
        self.tokenizer = tokenizer
        self.classifier = nn.Linear(1280, 1)


    def forward(self, images, texts):

        image_features = self.image_model.encode_image(images).to(device)

        image_features = F.normalize(image_features, dim=-1)

        text_features = self.text_model.forward(texts, tokenizer)

        text_features = F.normalize(text_features, dim=-1).to(device)


        combined = torch.cat([image_features, text_features], dim=1)

        logits = self.classifier(combined)#.squeeze(1)

        return logits

In [7]:
class MemeDataset(Dataset):
    def __init__(self, dataframe, image_folder, preprocess):
        self.df = dataframe
        self.image_folder = image_folder
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_folder, row['file_name'])
        image = Image.open(img_path).convert('RGB')
        image = self.preprocess(image)
        text = row['Text Transcription']
        label = torch.tensor(row['misogynous']).float()
        return image, text, label

In [8]:
from multilingual_clip import Config_MCLIP
import transformers
import torch

class MultilingualCLIP(transformers.PreTrainedModel):
    config_class = Config_MCLIP.MCLIPConfig

    def __init__(self, config, *args, **kwargs):
        super().__init__(config, *args, **kwargs)
        self.transformer = transformers.AutoModel.from_pretrained(config.modelBase, cache_dir=kwargs.get("cache_dir"))
        self.LinearTransformation = torch.nn.Linear(in_features=config.transformerDimensions,
                                                    out_features=config.numDims)

    def forward(self, txt, tokenizer):
        txt_tok = tokenizer(txt, padding=True, return_tensors='pt')
        text_tok = txt_tok.to(device)
        embs = self.transformer(**txt_tok)[0]
        att = txt_tok['attention_mask']
        embs = (embs * att.unsqueeze(2)).sum(dim=1) / att.sum(dim=1)[:, None]
        return self.LinearTransformation(embs)

    @classmethod
    def _load_state_dict_into_model(cls, model, state_dict, pretrained_model_name_or_path, _fast_init=True):
        model.load_state_dict(state_dict)
        return model, [], [], []

/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [9]:
from multilingual_clip import Config_MCLIP
import transformers
import torch

class MultilingualCLIP(transformers.PreTrainedModel):
    config_class = Config_MCLIP.MCLIPConfig

    def __init__(self, config, *args, **kwargs):
        super().__init__(config, *args, **kwargs)
        self.transformer = transformers.AutoModel.from_pretrained(config.modelBase, cache_dir=kwargs.get("cache_dir"))
        self.LinearTransformation = torch.nn.Linear(in_features=config.transformerDimensions,
                                                    out_features=config.numDims)

    def forward(self, txt, tokenizer):
        txt_tok = tokenizer(txt, padding=True, return_tensors='pt')
        text_tok = txt_tok.to(device)
        embs = self.transformer(**txt_tok)[0]
        att = txt_tok['attention_mask']
        embs = (embs * att.unsqueeze(2)).sum(dim=1) / att.sum(dim=1)[:, None]
        return self.LinearTransformation(embs)

    @classmethod
    def _load_state_dict_into_model(cls, model, state_dict, pretrained_model_name_or_path, _fast_init=True):
        model.load_state_dict(state_dict)
        return model, [], [], []

In [10]:
#IMAGE
image_model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16-plus-240', pretrained="laion400m_e32")
image_model.to(device)

#TEXT MODEL
model_name = 'M-CLIP/XLM-Roberta-Large-Vit-B-16Plus'

# Load Model & Tokenizer
text_model = MultilingualCLIP.from_pretrained(model_name, use_safetensors=True).to(device)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/834M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/221 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/399 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [11]:
model = MultimodalClassifier(image_model, text_model, tokenizer).to(device)

# Load finetuned weights
#model.load_state_dict(torch.load("/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/Dati/MODEL_SALIENCY_MAPS/multimodal_last_layer_fine.pt", map_location=device))
model.load_state_dict(torch.load("/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/Dati/MODEL_SALIENCY_MAPS/multimodal_full_fine.pt", map_location=device))
#model.load_state_dict(torch.load("/scratch_share/mind/claudia/multimodal_last_layer_fine.pt", map_location=device))

# Vai in modalità eval
model.eval()

MultimodalClassifier(
  (image_model): CLIP(
    (visual): VisionTransformer(
      (conv1): Conv2d(3, 896, kernel_size=(16, 16), stride=(16, 16), bias=False)
      (patch_dropout): Identity()
      (ln_pre): LayerNorm((896,), eps=1e-05, elementwise_affine=True)
      (transformer): Transformer(
        (resblocks): ModuleList(
          (0-11): 12 x ResidualAttentionBlock(
            (ln_1): LayerNorm((896,), eps=1e-05, elementwise_affine=True)
            (attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=896, out_features=896, bias=True)
            )
            (ls_1): Identity()
            (ln_2): LayerNorm((896,), eps=1e-05, elementwise_affine=True)
            (mlp): Sequential(
              (c_fc): Linear(in_features=896, out_features=3584, bias=True)
              (gelu): GELU(approximate='none')
              (c_proj): Linear(in_features=3584, out_features=896, bias=True)
            )
            (ls_2): Identity()
         

In [12]:
texts = []
ids = []
for i in range(len(text_df)):
        #doc_id= text_df["file_name"][i]
        #text = text_df["full_text"][i]
        doc_id= text_df["MEME"][i]
        text = text_df["TEXT"][i]
        ids.append(doc_id)
        texts.append(text)

# GRADCAM

In [ ]:
def forward_hook(module, input, output):
    global activations
    activations = output.detach()
def full_backward_hook(module, grad_input, grad_output):
    global gradients
    gradients = grad_output[0]

In [ ]:
activations = None
gradients = None

# Register hooks
target_layer = model.image_model.visual.transformer.resblocks[-9].ln_1


target_layer.register_forward_hook(forward_hook)
target_layer.register_full_backward_hook(full_backward_hook)



```
# Questo è formattato come codice
```

Controllo n° layer

In [ ]:
print(model.image_model.visual.transformer)

Transformer(
  (resblocks): ModuleList(
    (0-11): 12 x ResidualAttentionBlock(
      (ln_1): LayerNorm((896,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=896, out_features=896, bias=True)
      )
      (ls_1): Identity()
      (ln_2): LayerNorm((896,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (c_fc): Linear(in_features=896, out_features=3584, bias=True)
        (gelu): GELU(approximate='none')
        (c_proj): Linear(in_features=3584, out_features=896, bias=True)
      )
      (ls_2): Identity()
    )
  )
)


In [ ]:
activations = None
gradients = None

target_layer = model.image_model.visual.transformer.resblocks[-9].ln_1  #modificare in base al layer in cui si vuole lavorare
target_layer.register_forward_hook(forward_hook)
target_layer.register_full_backward_hook(full_backward_hook)


In [ ]:
def compute_gradcam(image_tensor, text_embedding):

    '''new part to adapt to PATCH VIT'''
    model.image_model.zero_grad()


    image_tensor = image_tensor.to(device).requires_grad_()


    # forward  to get image embedding  (non in no_grad)
    image_features = model.image_model.encode_image(image_tensor)          # [1, dim]
    image_features = F.normalize(image_features, dim=-1)


    similarity = F.cosine_similarity(image_features, text_embedding, dim=-1)  # shape [1]
    similarity_scalar = similarity.sum()

    # backward pass
    similarity_scalar.backward(retain_graph=True)

    global activations, gradients
    if activations is None:
        raise RuntimeError("Activations hook did not run. Check target layer of forward_hook.")
    if gradients is None:
        raise RuntimeError("Gradients hook did not run. Check target layer or PyTorch version (register_full_backward_hook).")

    # Per ViT: no token CLS (indice 0)
    # activations: [B, num_patches+1, hidden_dim]
    # gradients:   [B, num_patches+1, hidden_dim]
    patch_act = activations[:, 1:, :].detach()   # [1, N, D]
    patch_grad = gradients[:, 1:, :].detach()    # [1, N, D]


    weights = patch_grad.mean(dim=2, keepdim=True)   # [1, N, 1]
    cam = (weights * patch_act).sum(dim=2).squeeze(0) # [N]
    cam = F.relu(cam)
    # reshape in patch matrix
    grid_size = model.image_model.visual.image_size[0] // image_model.visual.patch_size[0]

    cam = cam.reshape(grid_size, grid_size).cpu().numpy()
    cam = cv2.resize(cam, (768, 768))

    return cam, similarity_scalar.item()


In [ ]:
def show_cam_and_ig_on_image(img, combined, meme):
    plt.figure(figsize=(8, 8), dpi=96)
    plt.imshow(img)
    plt.imshow(combined, cmap='jet', alpha=0.4)
    plt.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

    save_path = os.path.join(output_folder, f"{meme}_gradcam_and_ig.jpg")
    plt.savefig(save_path)
    plt.close()


In [ ]:
def show_cam(img, cam, img_name):

    meme = img_name.replace(".jpg", "")
    plt.figure(figsize=(8, 8), dpi=96)
      # plt.imshow(img)
    plt.imshow(cam, cmap='jet', alpha=0.4)
    plt.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

    save_path = os.path.join(output_folder, f"{meme}_gradcam.jpg")
    plt.savefig(save_path)
    plt.close()


In [ ]:
def integrated_gradients(text, model, tokenizer, image_embedding, steps=50):
    """
    Calculate Integrated Gradients for text relative to image.

    - Use baseline = zero embedding
    - Interpolate between baseline and input embedding (steps points)
    - Integrate gradients along the path (cumulative gradient)
    - Return normalized token importance
    """
    print(f"Integrated Gradients per text: {text[:30]}...")

    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device) #tokenization
    input_ids = inputs["input_ids"]   #id input
    attention_mask = inputs["attention_mask"]  #attention mask

    # Starting Embedding (real input)
    input_embeds = model.transformer.embeddings.word_embeddings(input_ids).detach().requires_grad_()

    # Baseline = embedding nullo
    baseline = torch.zeros_like(input_embeds).to(device) #neutral version

    grads = []
    for alpha in torch.linspace(0, 1, steps):
        # Baseline and input interpolation
        interpolated = baseline + alpha * (input_embeds - baseline)
        interpolated.retain_grad()

        # Forward pass con interpolated embedding
        outputs = model.transformer(inputs_embeds=interpolated, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :] #take cls that represent text
        projected = model.LinearTransformation(cls_embedding) # projection on shared space

        projected_norm = F.normalize(projected, dim=-1)
        image_norm = F.normalize(image_embedding, dim=-1)

        # Image-text similarity
        similarity = (projected_norm * image_norm).sum()
        model.zero_grad()
        similarity.backward()

        #Save gradients
        grads.append(interpolated.grad.detach().clone())

  # Average of gradients→ cumulative gradient
    avg_grads = torch.mean(torch.stack(grads), dim=0)
    attributions = (input_embeds - baseline) * avg_grads

    # Token importance (L2 norm, only positive contribution with ReLU)
    token_importance = F.relu(attributions).norm(dim=-1).squeeze(0).detach().cpu().numpy()


    # Normalization in [0,1] for comparison with Grad-CAM
    if token_importance.max() > 0:
        token_importance = (token_importance - token_importance.min()) / (token_importance.max() - token_importance.min() + 1e-8)

    # Textual tokens
    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))
    print(f"End Integrated Gradients for text: {text[:30]}...")
    return tokens, token_importance

In [ ]:
def append_token_importance_to_html(tokens, scores, html_file, header=""):
    jet = matplotlib.colormaps.get_cmap("Greens")

    html = ""
    if header:
        html += f"<h3>{header}</h3>\n"
    html += "<p style='font-family:monospace;'>"
    for token, score in zip(tokens, scores):
        rgb = jet(score)[:3]
        r, g, b = [int(255 * c) for c in rgb]
        token_clean = token.replace("▁", " ")
        html += f"<span style='background-color: rgb({r},{g},{b}); padding:2px; margin:1px;'>{token_clean}</span> "
    html += "</p>\n"
    with open(html_file, "a", encoding="utf-8") as f:
        f.write(html)

In [ ]:
def overlay_text_gradients_on_image(image, model_tokens, token_scores, reader, sigma=20, alpha=0.4, cmap_name='jet'):
    """
    Creates a 2D heatmap from token gradients (via OCR bounding boxes),
    blurs it with a Gaussian filter, and overlays it on the image with alpha=0.4
    using the specified colormap.

    Args:
        image: PIL.Image
        model_tokens: list of model tokens
        token_scores: array of token scores (IG or other importance)
        reader: EasyOCR Reader already instantiated
        sigma: standard deviation of the Gaussian filter
        alpha: transparency of the overlay
        cmap_name: matplotlib colormap (‘jet’, ‘Greens’, etc.)

    Returns:
        blended: PIL.Image with overlay
        heatmap: normalized numpy array [0,1]
    """

    H, W = image.size[1], image.size[0]
    heatmap = np.zeros((H, W), dtype=np.float32)

    # OCR on image
    ocr_results = reader.readtext(np.array(image))  # [(bbox, text, conf), ...]

    # Normalization token score
    scores_norm = (token_scores - token_scores.min()) / (token_scores.max() - token_scores.min() + 1e-8)

    associations = []  # to debug / check match

    for bbox, ocr_text, _ in ocr_results:
        ocr_subtokens = tokenizer.tokenize(ocr_text)
        N = len(ocr_subtokens)
        if N == 0:
            continue

        # Bounding box OCR
        bbox = np.array(bbox)
        xmin, ymin = bbox[:,0].min(), bbox[:,1].min()
        xmax, ymax = bbox[:,0].max(), bbox[:,1].max()
        width_per_token = (xmax - xmin) / N

        for idx, ocr_token in enumerate(ocr_subtokens):
            clean_ocr_token = ocr_token.replace("▁", "").lower()

            # Match with text token of the model
            score = 0
            for i, model_token in enumerate(model_tokens):
                clean_model_token = model_token.replace("▁", "").lower()
                if clean_ocr_token == clean_model_token:
                    score = scores_norm[i]
                    associations.append((ocr_token, model_token, score))
                    break  # take first match

            # Sotto-bbox for token
            x0 = int(xmin + idx * width_per_token)
            x1 = int(xmin + (idx + 1) * width_per_token)
            token_bbox = np.array([[x0, ymin], [x1, ymin], [x1, ymax], [x0, ymax]])

            # mask and add score
            mask = np.zeros_like(heatmap, dtype=np.uint8)
            cv2.fillPoly(mask, [token_bbox.astype(np.int32)], 1)
            heatmap += mask * score


    # Normalization [0,1]
    heatmap_norm = heatmap / (heatmap.max() + 1e-8)

    # Apply  colormap and blend with original image
    cmap = matplotlib.colormaps.get_cmap(cmap_name)
    overlay_map = (cmap(heatmap_norm)[..., :3] * 255).astype(np.uint8)
    blended = cv2.addWeighted(np.array(image), 1 - alpha, overlay_map, alpha, 0)
    return Image.fromarray(blended), heatmap


# Main loop

In [ ]:
# EasyOCR
reader = easyocr.Reader(['it','en'], gpu=torch.cuda.is_available())


html_file = os.path.join(output_folder, "all_text_gradients.html")
with open(html_file, "w", encoding="utf-8") as f:
    f.write("<html><body><h1>Token Importance</h1>\n")

for doc_id, text in zip(ids, texts):

    img_path = os.path.join(image_folder, doc_id)
    print(f"[INFO] Elaboro: {doc_id}")

    image = Image.open(img_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0).to(device)
    meme_name = doc_id.replace(".jpg", "")

    with torch.no_grad():
        image_embedding = model.image_model.encode_image(image_tensor)
        image_embedding = F.normalize(image_embedding, dim=-1)

    # Integrated Gradients for text
    tokens, token_scores = integrated_gradients(text, model.text_model, tokenizer, image_embedding, steps=50)

    overlay_img, heatmap_ig = overlay_text_gradients_on_image(image, tokens, token_scores, reader, cmap_name='jet')

    heatmap_ig_norm = heatmap_ig / (heatmap_ig.max())

    # Embedding CLS for GradCAM
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    outputs = model.text_model.transformer(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
    cls_embedding = outputs.last_hidden_state[:, 0, :]
    cls_embedding = model.text_model.LinearTransformation(cls_embedding)
    cls_embedding = F.normalize(cls_embedding, dim=-1)

    cam, sim = compute_gradcam(image_tensor, cls_embedding)
    cam_norm = cam / (cam.max())


    combined_matrix = cam_norm+heatmap_ig_norm
    combined_matrix = gaussian_filter(combined_matrix, sigma=20)

    combined_matrix = combined_matrix / (combined_matrix.max())
    show_cam_and_ig_on_image(image,combined_matrix, meme_name)

    np.save(os.path.join(output_folder, f"{meme_name}_gradcam_and_ig.npy"), combined_matrix)

    append_token_importance_to_html(tokens, token_scores, html_file, header=doc_id)

with open(html_file, "a", encoding="utf-8") as f:
    f.write("</body></html>")


[INFO] Elaboro: meme_1654.jpg
Integrated Gradients per text: HAI UNA RAGAZZA CHE NON TIENE ...
End Integrated Gradients for text: HAI UNA RAGAZZA CHE NON TIENE ...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[INFO] Elaboro: meme_0011.jpg
Integrated Gradients per text: QUANDO LA PERSONA CHE NON SOPP...
End Integrated Gradients for text: QUANDO LA PERSONA CHE NON SOPP...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[INFO] Elaboro: meme_0228.jpg
Integrated Gradients per text: QUANDO A 25 ANNI NON FAI SERAT...
End Integrated Gradients for text: QUANDO A 25 ANNI NON FAI SERAT...


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[INFO] Elaboro: meme_0960.jpg
Integrated Gradients per text: FRITTATA IN ARRIVO...
End Integrated Gradients for text: FRITTATA IN ARRIVO...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[INFO] Elaboro: meme_0913.jpg
Integrated Gradients per text: QUANDO GUIDA LEI...
End Integrated Gradients for text: QUANDO GUIDA LEI...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


LAVORO SULLE HEAD (Pizzo Davide)

In [ ]:
#numero head per layer?
print(model.image_model.visual.transformer.resblocks[-9].attn.num_heads)
print(model.image_model.visual.transformer.resblocks[-9].attn.embed_dim)

14
896


In [ ]:

#Calcolo metriche (codice identico a quello già implementato da Claudia,solo che risultava piu comodo averlo anche qua per le head)

def NSS(pred_map, gt_fix):
    """Normalized Scanpath Saliency"""
    pred_z = (pred_map - np.mean(pred_map)) / (np.std(pred_map) + 1e-8)
    return np.mean(pred_z[gt_fix > 0])


def EMD_2d(pred_map, gt_map, size=64):
    """Earth Mover's Distance"""
    # resize of the two maps
    cam_small = cv2.resize(pred_map, (size, size))
    gt_map_small = cv2.resize(gt_map, (size, size))

    # Flatten -> vectors
    a = cam_small.ravel().astype(np.float64)
    b = gt_map_small.ravel().astype(np.float64)

    # Normalization on probability distribution
    a /= (a.sum() + 1e-8)
    b /= (b.sum() + 1e-8)

    # matrix of coordinates (pixel)
    x, y = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    coords = np.stack([x.ravel(), y.ravel()], axis=1)

    # Cost matrix = euclidean distance in pixel
    M = ot.dist(coords, coords, metric="euclidean")

    # EMD = minimal cost to transform a in b
    return ot.emd2(a, b, M)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.8 MB/s eta 0:00:00


In [ ]:
def set_heatmap(df, offset_x, offset_y, base_heatmap):
  heatmap=base_heatmap.copy()
  for _, row in df.iterrows():
      x = int(round(row['CURRENT_FIX_X'] - offset_x))
      y = int(round(row['CURRENT_FIX_Y'] - offset_y))
      duration = row['CURRENT_FIX_DURATION']
      if 0 <= x < 768 and 0 <= y < 768:
        heatmap[y,x] += duration   #inverse because of the matrix [row=height, column=width]

  heatmap = gaussian_filter(heatmap, sigma=20)  # to blur the image
  return heatmap
def set_values():
  screen_width, screen_height=1920,1080  # 4:3
  meme_width, meme_height=768,768

  offset_x = (screen_width - meme_width) // 2
  offset_y = (screen_height - meme_height) // 2

  heatmap = np.zeros((meme_height, meme_width), dtype=float)
  return offset_x, offset_y, heatmap

In [ ]:
input_to_attn = None #var globale necessaria per depositare l'input dell'attention

def pre_hook(module, input):
    global input_to_attn
    input_to_attn = input[0].detach()

target_attn = model.image_model.visual.transformer.resblocks[-9].attn  #cambia l'indice per analizzare una profondità diversa
target_attn.register_forward_pre_hook(pre_hook) #registra hook
print("Hook registrato")

Hook registrato!


In [ ]:
# SETUP: COMMENTA RIGHE IN BASE ALLA TIPOLOGIA DI MODELLO FINE-TUNATO CHE STAI UTILIZZANDO
reader = easyocr.Reader(['it','en'], gpu=torch.cuda.is_available())

offset_x, offset_y, base_heatmap = set_values()
df_meme_agg_folder = '/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/Dati/DF_MEME_AGGREGATED'

#output_csv = '/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/fine_grained/metrics_heads[-11].csv'
#os.makedirs('/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/fine_grained', exist_ok=True)
output_csv = '/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/full_fine/metrics_heads[-11].csv'
os.makedirs('/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/full_fine', exist_ok=True)

results = []

# loop (eseguibile su colab,ma ci metterà un pò)
contatore=0 #per tenere traccia del processo
for doc_id, text in zip(ids, texts):  # tutti i 96 meme
    print(f"[INFO] Elaboro: {doc_id}")

    # Carica immagine
    img_path = os.path.join(image_folder, doc_id)
    image = Image.open(img_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0).to(device)

    # heatmap umana : ground truth dall'eye-tracking
    meme_aggregated = pd.read_csv(
        os.path.join(df_meme_agg_folder, f"{doc_id}.csv")
    )
    human_heatmap = set_heatmap(meme_aggregated, offset_x, offset_y, base_heatmap) #produzione heatmap umana

    #Mappa punti fissazione,serve percalcolare metrica NSS
    gt_fix = np.zeros((768, 768))
    for _, row in meme_aggregated.iterrows():
        x = int(round(row['CURRENT_FIX_X'] - offset_x))
        y = int(round(row['CURRENT_FIX_Y'] - offset_y))
        if 0 <= x < 768 and 0 <= y < 768:
            gt_fix[y, x] = 1

    gt_map = human_heatmap.copy()
    gt_map = (gt_map - gt_map.min()) / (gt_map.max() + 1e-8)

    #calcolo IG
    with torch.no_grad():
        image_embedding = model.image_model.encode_image(image_tensor)
        image_embedding = F.normalize(image_embedding, dim=-1)

    tokens, token_scores = integrated_gradients(
        text, model.text_model, tokenizer, image_embedding, steps=50
    )
    #per ottenere una heatmap testuale che sia sovrapponibile a quella visiva
    _, heatmap_ig = overlay_text_gradients_on_image(
        image, tokens, token_scores, reader, cmap_name='jet'
    )
    heatmap_ig_norm = heatmap_ig / (heatmap_ig.max() + 1e-8)

    # attentio delle head
    with torch.no_grad():
        _ = model.image_model.encode_image(image_tensor)
        _, attn_weights = target_attn(
            input_to_attn, input_to_attn, input_to_attn,
            need_weights=True,
            average_attn_weights=False
        )

    cls_attention = attn_weights[0, :, 0, 1:]  # [14, 225]
    head_heatmaps = cls_attention.reshape(14, 15, 15).cpu().numpy() #reshape a 15x15=225 patch

    # NSS ed EMD per ogni head
    for head_idx in range(14):
      #normalizzazione
        heatmap_visual = head_heatmaps[head_idx]
        heatmap_visual = (heatmap_visual - heatmap_visual.min()) / \
                         (heatmap_visual.max() - heatmap_visual.min() + 1e-8)
        heatmap_visual = cv2.resize(heatmap_visual, (768, 768))
        #saliency totale: visiva+testuale; poi smoothing e normalizzazione per avere heatmap confrontabile con grount truth umana
        combined = heatmap_visual + heatmap_ig_norm
        combined = gaussian_filter(combined, sigma=20)
        combined = combined / (combined.max() + 1e-8)

        nss = NSS(combined, gt_fix)
        emd = EMD_2d(combined, gt_map)

        results.append({
            'meme': doc_id,
            'head': head_idx,
            'NSS': nss,
            'EMD': emd
        })

    print(f"[OK] {doc_id} — 14 head elaborate")
    contatore += 1
    print(f"Meme elaborati: {contatore}/{len(ids)}")

#SALVA CSV
df_results = pd.DataFrame(results)
df_results.to_csv(output_csv, index=False)
print(f"Salvato: {output_csv}")

[INFO] Elaboro: meme_1654.jpg
Integrated Gradients per text: HAI UNA RAGAZZA CHE NON TIENE ...
End Integrated Gradients for text: HAI UNA RAGAZZA CHE NON TIENE ...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[OK] meme_1654.jpg — 14 head elaborate
Meme elaborati: 1/96
[INFO] Elaboro: meme_0011.jpg
Integrated Gradients per text: QUANDO LA PERSONA CHE NON SOPP...
End Integrated Gradients for text: QUANDO LA PERSONA CHE NON SOPP...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[OK] meme_0011.jpg — 14 head elaborate
Meme elaborati: 2/96
[INFO] Elaboro: meme_0228.jpg
Integrated Gradients per text: QUANDO A 25 ANNI NON FAI SERAT...
End Integrated Gradients for text: QUANDO A 25 ANNI NON FAI SERAT...


...

[OK] meme_0437.jpg — 14 head elaborate
Meme elaborati: 94/96
[INFO] Elaboro: meme_0960.jpg
Integrated Gradients per text: FRITTATA IN ARRIVO...
End Integrated Gradients for text: FRITTATA IN ARRIVO...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[OK] meme_0960.jpg — 14 head elaborate
Meme elaborati: 95/96
[INFO] Elaboro: meme_0913.jpg
Integrated Gradients per text: QUANDO GUIDA LEI...
End Integrated Gradients for text: QUANDO GUIDA LEI...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[OK] meme_0913.jpg — 14 head elaborate
Meme elaborati: 96/96
Salvato: /content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/full_fine/metrics_heads[-11].csv


#### Qua solo codice per controllare alcune piccole cose (opzionale)

In [ ]:
with torch.no_grad():
    _, attn_weights = target_attn(
        input_to_attn, input_to_attn, input_to_attn,
        need_weights=True,
        average_attn_weights=False
    )

print("Attention weights shape:", attn_weights.shape) #controllo se tutto corretto

Attention weights shape: torch.Size([1, 14, 226, 226])


In [ ]:
# perr ogni head prendi l'attenzione del CLS verso le patch
# attn_weights: [1, 14, 226, 226]
cls_attention = attn_weights[0, :, 0, 1:]  # [14, 225] — escludi CLS come destinazione

print("CLS attention shape:", cls_attention.shape)
# [14, 225] → 14 head, ognuna con 225 valori (uno per patch)

grid_size = 15  # 225 = 15x15 reshape griglia
head_heatmaps = cls_attention.reshape(14, grid_size, grid_size).cpu().numpy()
print("Head heatmaps shape:", head_heatmaps.shape)
# [14, 15, 15]

CLS attention shape: torch.Size([14, 225])
Head heatmaps shape: (14, 15, 15)


## PLOT RISULTATI

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os


#df = pd.read_csv('/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/fine_grained/metrics_heads[-11].csv')
#output_dir = '/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/fine_grained/'

df = pd.read_csv('/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/full_fine/metrics_heads[-11].csv')
output_dir = '/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/full_fine/'

os.makedirs(output_dir, exist_ok=True)
#commenta in base a tipologia di modello finetunato

# head su asse X
head_order = list(range(14))

for metric in ['NSS', 'EMD']:
    fig, ax = plt.subplots(figsize=(16, 6))

    sns.boxplot(
        data=df, x='head', y=metric,
        order=head_order,
        color='plum' if metric == 'NSS' else 'gold',
        ax=ax
    )

    # media
    means = df.groupby('head')[metric].mean()
    for head_idx in head_order:
        mean_val = means[head_idx]
        ax.text(
            head_idx,
            ax.get_ylim()[1] * 0.95,
            f'{mean_val:.2f}',
            ha='center', va='top',
            fontsize=9, fontweight='bold',
            color='black'
        )

    ax.set_title(f'{metric} per Head — 2° Layer (Full Fine-Tuned)',
                 fontsize=16, fontweight='bold')
    ax.set_xlabel('Head', fontsize=14)
    ax.set_ylabel(metric, fontsize=14)
    ax.set_xticks(head_order)
    ax.set_xticklabels([f'H{i}' for i in head_order], fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{metric}_heads_layer-11.png'), dpi=300)
    plt.show()
    print(f"Salvato {metric}_heads_layer-11.png")

## Salvataggio 96 heatmap per miglior layerxhead; solo heatmap non file.npy

In [ ]:

#miglior combinazione layerXhead,ti basta modificare questi due per adattare tutto il codice della cella
LAYER_IDX = -9      # 4° layer
HEAD_IDX  = 3       # 4ª head

output_best = '/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/full_fine/best_layer[-9]_head4'
#output_best = '/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/fine_grained/best_layer[-9]_head4'

os.makedirs(output_best, exist_ok=True)

reader = easyocr.Reader(['it','en'], gpu=torch.cuda.is_available())

#hook
input_to_attn = None
def pre_hook(module, inp):
    global input_to_attn
    input_to_attn = inp[0].detach()

target_attn = model.image_model.visual.transformer.resblocks[LAYER_IDX].attn
handle = target_attn.register_forward_pre_hook(pre_hook)

# loop 96 meme
contatore = 0 #contatore per monitorare il procedimento,su colab ci vorrà un po di tempo
for doc_id, text in zip(ids, texts):
    img_path = os.path.join(image_folder, doc_id)
    image = Image.open(img_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0).to(device)
    meme_name = doc_id.replace(".jpg", "")

    # Integrated Gradients
    with torch.no_grad():
        image_embedding = model.image_model.encode_image(image_tensor)
        image_embedding = F.normalize(image_embedding, dim=-1)

    tokens, token_scores = integrated_gradients(
        text, model.text_model, tokenizer, image_embedding, steps=50
    )
    _, heatmap_ig = overlay_text_gradients_on_image(
        image, tokens, token_scores, reader, cmap_name='jet'
    )
    heatmap_ig_norm = heatmap_ig / (heatmap_ig.max() + 1e-8)

    # Attention della head selezionata
    with torch.no_grad():
        _ = model.image_model.encode_image(image_tensor)
        _, attn_weights = target_attn(
            input_to_attn, input_to_attn, input_to_attn,
            need_weights=True, average_attn_weights=False
        )
    cls_attention = attn_weights[0, :, 0, 1:]
    head_heatmaps = cls_attention.reshape(14, 15, 15).cpu().numpy()

    # fusione heatmap per la head selezionata (stessa  roba che si faceva in precedenza)
    heatmap_visual = head_heatmaps[HEAD_IDX]
    heatmap_visual = (heatmap_visual - heatmap_visual.min()) / \
                     (heatmap_visual.max() - heatmap_visual.min() + 1e-8)
    heatmap_visual = cv2.resize(heatmap_visual, (768, 768))

    combined = heatmap_visual + heatmap_ig_norm
    combined = gaussian_filter(combined, sigma=20)
    combined = combined / (combined.max() + 1e-8)

    #salva
    image_768 = image.resize((768, 768))
    plt.figure(figsize=(8, 8))
    plt.imshow(image_768)
    plt.imshow(combined, cmap='jet', alpha=0.4)
    plt.axis('off')
    plt.title(f"{meme_name} — Layer {LAYER_IDX} · Head {HEAD_IDX+1}", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(output_best, f"{meme_name}_L[{LAYER_IDX}]_H{HEAD_IDX+1}.png"),
                dpi=150, bbox_inches='tight')
    plt.close()

    #np.save(os.path.join(output_best, f"{meme_name}_L[{LAYER_IDX}]_H{HEAD_IDX+1}.npy"), combined)  QUESTI VALORI LI HO GIA

    contatore += 1
    print(f"Meme elaborati: {contatore}/{len(ids)}")

handle.remove()
print(f"\nheatmap salvate in {output_best}")

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteIntegrated Gradients per text: HAI UNA RAGAZZA CHE NON TIENE ...
End Integrated Gradients for text: HAI UNA RAGAZZA CHE NON TIENE ...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Meme elaborati: 1/96
Integrated Gradients per text: QUANDO LA PERSONA CHE NON SOPP...
End Integrated Gradients for text: QUANDO LA PERSONA CHE NON SOPP...
Meme elaborati: 2/96
Integrated Gradients per text: QUANDO A 25 ANNI NON FAI SERAT...
End Integrated Gradients for text: QUANDO A 25 ANNI NON FAI SERAT...
Meme elaborati: 3/96
Integrated Gradients per text: CHE MI PREPARO PER FARE SERATA...
End Integrated Gradients for text: CHE MI PREPARO PER FARE SERATA...
Meme elaborati: 4/96
Integrated Gradients per text: NON IMPORTA SE CI SONO 40 GRAD...
End Integrated Gradients for text: NON IMPORTA SE CI SONO 40 GRAD...
Meme elaborati: 5/96
Integrated Gradients per text: QUANDO DOPO TRA CENA E BENZINA...
End Integrated Gradients for text: QUANDO DOPO TRA CENA E BENZINA...
Meme elaborati: 6/96
Integrated Gradients per text: QUANDO GI SONO GRADI, HAI LA P...
End Integrated Gradients for text: QUANDO GI SONO GRADI, HAI LA P...
Meme elaborati: 7/96
Integrated Gradients per text: SONO MOLTO CONTENT

### Quali sono i migliori e peggiori meme,nella combinazione analizzata,in base a entrambe le metriche?

In [ ]:
#df = pd.read_csv('/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/full_fine/metrics_heads[-9].csv')
df = pd.read_csv('/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/fine_grained/metrics_heads[-9].csv')
#commenta in base al modello finetunato

df_h = df[df['head'] == 3]

print(f"Righe (meme) per head 3: {len(df_h)}\n")

#NSS (più è alto meglio è)
best_nss  = df_h.loc[df_h['NSS'].idxmax()]
worst_nss = df_h.loc[df_h['NSS'].idxmin()]
print("NSS")
print(f"  Migliore: {best_nss['meme']}  →  NSS = {best_nss['NSS']:.4f}")
print(f"  Peggiore: {worst_nss['meme']}  →  NSS = {worst_nss['NSS']:.4f}\n")

#EMD (più basso è,meglio è)
best_emd  = df_h.loc[df_h['EMD'].idxmin()]
worst_emd = df_h.loc[df_h['EMD'].idxmax()]
print("EMD")
print(f"  Migliore: {best_emd['meme']}  →  EMD = {best_emd['EMD']:.4f}")
print(f"  Peggiore: {worst_emd['meme']}  →  EMD = {worst_emd['EMD']:.4f}")

Righe (meme) per head 3: 96

NSS
  Migliore: meme_0388.jpg  →  NSS = 2.1427
  Peggiore: meme_1856.jpg  →  NSS = 0.6292

EMD
  Migliore: meme_1265.jpg  →  EMD = 3.1221
  Peggiore: meme_0388.jpg  →  EMD = 13.1143


In [13]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ----- CONFIG -----
LAYER_IDX = -9          # il blocco da visualizzare
DOC_ID    = ids[0]      # il meme (cambialo se ne vuoi un altro)

# hook per catturare l'input dell'attention
input_to_attn = None
def pre_hook(module, inp):
    global input_to_attn
    input_to_attn = inp[0].detach()

target_attn = model.image_model.visual.transformer.resblocks[LAYER_IDX].attn
handle = target_attn.register_forward_pre_hook(pre_hook)

image = Image.open(os.path.join(image_folder, DOC_ID)).convert("RGB")
image_tensor = preprocess(image).unsqueeze(0).to(device)

with torch.no_grad():
    _ = model.image_model.encode_image(image_tensor)
    _, attn_weights = target_attn(
        input_to_attn, input_to_attn, input_to_attn,
        need_weights=True, average_attn_weights=False
    )
handle.remove()

cls_attention = attn_weights[0, :, 0, 1:]              # [14, 225]
head_heatmaps = cls_attention.reshape(14, 15, 15).cpu().numpy()

# ----- GRIGLIA 2x7 -----
image_768 = image.resize((768, 768))
fig, axes = plt.subplots(2, 7, figsize=(28, 8))
axes = axes.flatten()

for h in range(14):
    hm = head_heatmaps[h]
    hm = (hm - hm.min()) / (hm.max() - hm.min() + 1e-8)
    hm = cv2.resize(hm, (768, 768))
    axes[h].imshow(image_768)
    axes[h].imshow(hm, cmap='jet', alpha=0.4)
    axes[h].set_title(f"Head {h+1}", fontsize=14)
    axes[h].axis('off')

plt.suptitle(f"Attention heads — layer {LAYER_IDX}", fontsize=18)
plt.tight_layout()
plt.savefig(f"attention_heads_layer{LAYER_IDX}.png", dpi=150, bbox_inches='tight')
plt.show()

Output hidden; open in https://colab.research.google.com to view.